# Quickstart — score a transcript in 5 cells

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kikolo3000/clinical-skill-architecture/blob/main/notebooks/01_quickstart.ipynb)

The **Gottschalk-Gleser depression scale** scores 7 facets of depressive content in *what someone says* — Hopelessness, Self-accusation, Psychomotor retardation, Somatic concerns, Death/mutilation, Separation, and Hostility outward. Trained human raters take ~30 minutes per session. This package replaces them with a Claude-based agent that scores in seconds and ships a per-clause rationale you can audit.

| Subscale | Captures                                                            |
|----------|---------------------------------------------------------------------|
| HOP      | Hopelessness, despair, futility, lack of motivation                 |
| SAC      | Self-accusation, guilt, shame, suicidal ideation                    |
| PMR      | Psychomotor retardation, fatigue, slowness                          |
| SOM      | Somatic concerns, bodily complaints, illness fears                   |
| DAM      | Death and mutilation references                                      |
| SEP      | Separation, abandonment, loss of relationships                       |
| HOS      | Hostility outward — anger, resentment, irritability                  |

Run cells in order; total runtime ~3 minutes, total cost ~$0.01.

In [ ]:
# Install (uncomment in Colab — locally use `pip install clinical-skill-architecture`)
# %pip install clinical-skill-architecture matplotlib

import getpass
import os

if not os.environ.get('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Anthropic API key: ')

In [ ]:
from csa import score

transcript = (
    "I: how have you been feeling about the future\n"
    "S: i just don't see the point anymore honestly. "
    "nothing is going to change no matter what i do. "
    "i've been going to work every day though."
)

result = score(transcript, model='claude-sonnet-4-6')

print(f'Total depression score: {result.total_depression:.2f}')
print(f'Word count:             {result.word_count}')
print(f'Estimated cost:         ${result.estimated_cost_usd:.4f}')

In [ ]:
import matplotlib.pyplot as plt

subs = list(result.subscales.values())
fig, ax = plt.subplots(figsize=(8, 3.5))
bars = ax.bar([s.subscale for s in subs], [s.score for s in subs],
              color=['#c0392b' if s.count > 0 else '#bdc3c7' for s in subs])
ax.set_ylabel('G-G score')
ax.set_title('Subscale scores (red = at least one clause coded)')
for bar, s in zip(bars, subs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{s.score:.2f}', ha='center', fontsize=9)
plt.tight_layout(); plt.show()

print('\nClauses coded by the agent:')
for c in result.codings:
    print(f'  [{c.subscale}/{c.sub_item}] perspective={c.perspective} weight={c.weight}')
    print(f'      "{c.clause}"')
    print(f'      → {c.rationale}\n')

## What just happened?

1. The transcript was split into one fragment per `S:` turn (here, one).
2. **Stage 1 (screening)** — the agent read the fragment, identified Hopelessness as the only flagged subscale.
3. **Stage 2 (detail)** — the agent loaded the HOP `SKILL.md` and scored the fragment clause by clause, ignoring the distractor ("i've been going to work every day though").
4. The G-G formula `sqrt((weighted_sum + 0.5) * 100 / word_count)` collapsed the codings into 7 subscale scores plus a total.

## Next

- **`02_synthetic_validation.ipynb`** — see the agent benchmarked against ground truth on three deliberately tricky cases (clear depression, distractor-heavy, and explicit denial), with a hierarchical accuracy plot.
- **`03_real_transcript.ipynb`** — a 15-turn realistic clinical-interview transcript scored end-to-end, with per-utterance and aggregate visualisations.
- **README** — install, configuration, citation, the paper, the disclaimer.